In [ ]:
import numpy as np
import os
import json
import pickle
import time
import datetime
import argparse
import matplotlib.pyplot as plt
from joblib import dump, load
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

# Set up logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('eeg_pipeline')

# Directory configuration
models_dir = '../models'
results_dir = os.path.join(models_dir, 'predictions')
os.makedirs(models_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

def load_data(models_dir='../models'):
    """
    Load preprocessed EEG data from the models directory
    """
    logger.info("Loading preprocessed data...")
    
    # Find the latest preprocessed files
    x_files = [f for f in os.listdir(models_dir) if f.startswith('X_preprocessed') and f.endswith('.npy')]
    y_files = [f for f in os.listdir(models_dir) if f.startswith('y_labels') and f.endswith('.npy')]
    
    if not x_files or not y_files:
        raise FileNotFoundError(f"Preprocessed data files not found in {models_dir}")
    
    # Sort by timestamp to get the latest
    x_files.sort(reverse=True)
    y_files.sort(reverse=True)
    
    X = np.load(os.path.join(models_dir, x_files[0]))
    y = np.load(os.path.join(models_dir, y_files[0]))
    
    logger.info(f"Loaded data: X shape {X.shape}, y shape {y.shape}")
    logger.info(f"Classes: {np.unique(y)}")
    
    # Try to load preprocessing info
    preprocessing_info = None
    info_files = [f for f in os.listdir(models_dir) if f.startswith('preprocessing_info') and f.endswith('.json')]
    if info_files:
        with open(os.path.join(models_dir, info_files[0]), 'r') as f:
            preprocessing_info = json.load(f)
    
    return X, y, preprocessing_info

def reshape_to_3d(X, n_channels=64):
    """
    Reshape the 2D EEG data to 3D format (epochs, channels, time)
    """
    n_samples = X.shape[0]
    n_times = X.shape[1] // n_channels
    return X.reshape(n_samples, n_channels, n_times).astype(np.float64)

def load_best_pipeline():
    """
    Load the best pipeline model from the optimization process
    """
    logger.info("Loading the best pipeline model...")
    pipeline_path = os.path.join(models_dir, 'best_pipeline.joblib')
    info_path = os.path.join(models_dir, 'best_pipeline_info.json')
    
    try:
        # Load the pipeline
        pipeline = load(pipeline_path)
        logger.info(f"Pipeline loaded from {pipeline_path}")
        
        # Load pipeline info
        if os.path.exists(info_path):
            with open(info_path, 'r') as f:
                pipeline_info = json.load(f)
            logger.info(f"Pipeline info loaded: {pipeline_info['best_pipeline_name']}")
            return pipeline, pipeline_info
        else:
            logger.warning("Pipeline info not found, creating minimal info")
            return pipeline, {'is_csp': False, 'best_pipeline_name': 'Unknown'}
            
    except Exception as e:
        logger.error(f"Error loading pipeline: {e}")
        return None, None

def train_pipeline(pipeline, X, y, X_3d=None, test_size=0.2, is_csp=False):
    """
    Train the pipeline with the provided data and save the trained model
    """
    logger.info("Splitting data into training and test sets...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )
    
    # For CSP, we need to prepare 3D data
    X_train_3d = None
    X_test_3d = None
    
    if is_csp and X_3d is not None:
        X_train_3d = X_3d[np.where(np.isin(range(len(X)), np.where(~np.isin(range(len(X)), X_test.index))[0]))[0]]
        X_test_3d = X_3d[np.where(np.isin(range(len(X)), X_test.index))[0]]
    
    # Check if it's a custom ensemble with both standard and CSP pipelines
    is_ensemble = hasattr(pipeline, 'standard_pipeline') and hasattr(pipeline, 'csp_pipeline')
    
    logger.info(f"Training pipeline (type: {'Ensemble' if is_ensemble else 'CSP' if is_csp else 'Standard'})...")
    
    start_time = time.time()
    
    try:
        if is_ensemble and X_train_3d is not None:
            pipeline.fit(X_train, y_train, X_3d=X_train_3d)
        elif is_csp and X_train_3d is not None:
            pipeline.fit(X_train_3d, y_train)
        else:
            pipeline.fit(X_train, y_train)
            
        training_time = time.time() - start_time
        logger.info(f"Training completed in {training_time:.2f} seconds")
        
        # Save the trained model
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        model_path = os.path.join(models_dir, f'final_model_{timestamp}.joblib')
        dump(pipeline, model_path)
        logger.info(f"Trained model saved to: {model_path}")
        
        # Also save as latest model
        latest_path = os.path.join(models_dir, 'final_model_latest.joblib')
        dump(pipeline, latest_path)
        logger.info(f"Trained model also saved as: {latest_path}")
        
        # Evaluate on test set
        logger.info("Evaluating on test set...")
        
        if is_ensemble and X_test_3d is not None:
            y_pred = pipeline.predict(X_test, X_3d=X_test_3d)
        elif is_csp and X_test_3d is not None:
            y_pred = pipeline.predict(X_test_3d)
        else:
            y_pred = pipeline.predict(X_test)
            
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        
        logger.info(f"Test Set Metrics:")
        logger.info(f"Accuracy: {accuracy:.4f}")
        logger.info(f"F1 Score: {f1:.4f}")
        logger.info(f"Precision: {precision:.4f}")
        logger.info(f"Recall: {recall:.4f}")
        
        # Save test set and predictions for later use
        np.save(os.path.join(models_dir, 'X_test.npy'), X_test)
        np.save(os.path.join(models_dir, 'y_test.npy'), y_test)
        np.save(os.path.join(models_dir, 'y_pred.npy'), y_pred)
        
        # Save training results
        training_info = {
            'timestamp': timestamp,
            'pipeline_type': 'Ensemble' if is_ensemble else 'CSP' if is_csp else 'Standard',
            'training_time': training_time,
            'test_metrics': {
                'accuracy': float(accuracy),
                'f1_score': float(f1),
                'precision': float(precision),
                'recall': float(recall)
            },
            'model_path': model_path,
            'test_set_size': len(y_test),
            'train_set_size': len(y_train),
            'confusion_matrix': confusion_matrix(y_test, y_pred).tolist()
        }
        
        with open(os.path.join(models_dir, f'training_info_{timestamp}.json'), 'w') as f:
            json.dump(training_info, f, indent=4)
            
        # Plot confusion matrix
        plt.figure(figsize=(10, 8))
        cm = confusion_matrix(y_test, y_pred)
        plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title('Confusion Matrix')
        plt.colorbar()
        
        classes = np.unique(y)
        tick_marks = np.arange(len(classes))
        plt.xticks(tick_marks, classes)
        plt.yticks(tick_marks, classes)
        
        # Add text annotations
        thresh = cm.max() / 2
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, format(cm[i, j], 'd'),
                        horizontalalignment="center",
                        color="white" if cm[i, j] > thresh else "black")
                
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig(os.path.join(results_dir, f'confusion_matrix_{timestamp}.png'))
        
        return pipeline, training_info
        
    except Exception as e:
        logger.error(f"Error during training: {e}")
        return None, None

def predict_realtime(pipeline, X, X_3d=None, y=None, is_csp=False, visualize=True, delay=0.1):
    """
    Simulate real-time prediction with the trained pipeline
    """
    n_samples = len(X)
    predictions = []
    times = []
    
    # Initialize visualization if requested
    if visualize:
        plt.figure(figsize=(12, 8))
        plt.ion()  # Interactive mode
        plt.show()
    
    logger.info(f"Starting real-time prediction simulation ({n_samples} samples)...")
    logger.info("Press Ctrl+C to stop the simulation")
    
    try:
        for i in range(n_samples):
            # Extract current sample
            X_sample = X[i:i+1]
            X_3d_sample = X_3d[i:i+1] if X_3d is not None else None
            
            # Make prediction
            start_time = time.time()
            
            # Check if it's a custom ensemble
            is_ensemble = hasattr(pipeline, 'standard_pipeline') and hasattr(pipeline, 'csp_pipeline')
            
            if is_ensemble and X_3d_sample is not None:
                y_pred = pipeline.predict(X_sample, X_3d=X_3d_sample)
                try:
                    y_proba = pipeline.predict_proba(X_sample, X_3d=X_3d_sample)
                except:
                    y_proba = None
            elif is_csp and X_3d_sample is not None:
                y_pred = pipeline.predict(X_3d_sample)
                try:
                    y_proba = pipeline.predict_proba(X_3d_sample)
                except:
                    y_proba = None
            else:
                y_pred = pipeline.predict(X_sample)
                try:
                    y_proba = pipeline.predict_proba(X_sample)
                except:
                    y_proba = None
                    
            pred_time = time.time() - start_time
            times.append(pred_time)
            
            predictions.append(int(y_pred[0]))
            
            # Calculate current accuracy if ground truth is available
            current_acc = None
            if y is not None:
                current_acc = accuracy_score(y[:i+1], predictions)
                
            # Display information
            class_pred = int(y_pred[0])
            logger.info(f"Sample {i+1}/{n_samples} - Predicted: {class_pred}" + 
                      (f", True: {y[i]}, Correct: {'✓' if class_pred == y[i] else '✗'}" if y is not None else "") +
                      f" - Time: {pred_time:.6f}s")
            
            # Visualize if requested
            if visualize and i % 5 == 0:  # Update visualization every 5 samples
                plt.clf()
                
                # Predictions graph
                plt.subplot(2, 1, 1)
                plt.plot(predictions, 'b-', linewidth=2)
                if y is not None:
                    plt.plot(y[:len(predictions)], 'r--', alpha=0.7)
                plt.title('Real-Time Predictions')
                plt.xlabel('Sample')
                plt.ylabel('Class')
                if y is not None:
                    plt.legend(['Predicted', 'Actual'])
                    
                # Add accuracy text if ground truth is available
                if current_acc is not None:
                    plt.text(0.02, 0.9, f"Accuracy: {current_acc:.4f}", 
                             transform=plt.gca().transAxes,
                             bbox=dict(facecolor='white', alpha=0.8))
                
                # Time histogram
                plt.subplot(2, 1, 2)
                plt.hist(times, bins=min(20, len(times)), color='skyblue', edgecolor='black')
                plt.axvline(np.mean(times), color='r', linestyle='--', 
                            label=f'Avg: {np.mean(times):.6f}s')
                plt.title('Prediction Times')
                plt.xlabel('Time (s)')
                plt.ylabel('Frequency')
                plt.legend()
                
                plt.tight_layout()
                plt.pause(0.01)
                
            # Simulate delay between samples
            time.sleep(delay)
            
    except KeyboardInterrupt:
        logger.info("Simulation interrupted by user")
    except Exception as e:
        logger.error(f"Error during simulation: {e}")
    finally:
        if visualize:
            plt.ioff()
        
        # Show summary
        if predictions:
            avg_time = np.mean(times)
            max_time = np.max(times)
            min_time = np.min(times)
            
            logger.info("\nSimulation Summary:")
            logger.info(f"Processed samples: {len(predictions)}/{n_samples}")
            logger.info(f"Average time: {avg_time:.6f}s")
            logger.info(f"Maximum time: {max_time:.6f}s")
            logger.info(f"Minimum time: {min_time:.6f}s")
            
            if y is not None:
                final_acc = accuracy_score(y[:len(predictions)], predictions)
                logger.info(f"Final accuracy: {final_acc:.4f}")
                
                if len(np.unique(y[:len(predictions)])) > 1:
                    f1 = f1_score(y[:len(predictions)], predictions, average='weighted')
                    logger.info(f"F1 score: {f1:.4f}")
            
            # Save results
            timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            results_file = os.path.join(results_dir, f'realtime_prediction_{timestamp}.json')
            
            results = {
                'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'predictions': predictions,
                'times': [float(t) for t in times],
                'summary': {
                    'samples': len(predictions),
                    'avg_time': float(avg_time),
                    'max_time': float(max_time),
                    'min_time': float(min_time)
                }
            }
            
            if y is not None:
                results['accuracy'] = float(final_acc)
                
            with open(results_file, 'w') as f:
                json.dump(results, f, indent=4)
                
            logger.info(f"Results saved to: {results_file}")
            
            # Plot final results
            if visualize:
                plt.figure(figsize=(12, 10))
                
                # Predictions graph
                plt.subplot(2, 1, 1)
                plt.plot(predictions, 'b-', linewidth=2)
                if y is not None:
                    plt.plot(y[:len(predictions)], 'r--', alpha=0.7)
                plt.title('Real-Time Predictions')
                plt.xlabel('Sample')
                plt.ylabel('Class')
                if y is not None:
                    plt.legend(['Predicted', 'Actual'])
                
                # Time histogram
                plt.subplot(2, 1, 2)
                plt.hist(times, bins=min(20, len(times)), color='skyblue', edgecolor='black')
                plt.axvline(avg_time, color='r', linestyle='--', 
                            label=f'Average: {avg_time:.6f}s')
                plt.title('Prediction Times')
                plt.xlabel('Time (s)')
                plt.ylabel('Frequency')
                plt.legend()
                
                plt.tight_layout()
                plt.savefig(os.path.join(results_dir, f'realtime_plot_{timestamp}.png'))
                plt.show()
                
            return results
        
        return None

def main():
    """
    Main function for training and prediction using optimized pipeline
    """
    parser = argparse.ArgumentParser(description='EEG Train and Predict with Optimized Pipeline')
    parser.add_argument('--mode', choices=['train', 'predict', 'both'], default='both',
                       help='Mode: train the model, predict with existing model, or both')
    parser.add_argument('--visualize', action='store_true', help='Visualize predictions in real-time')
    parser.add_argument('--delay', type=float, default=0.1, help='Delay between samples in seconds')
    parser.add_argument('--test_size', type=float, default=0.2, help='Test set size for training')
    
    args = parser.parse_args()
    
    try:
        # Load preprocessed data
        X, y, preprocessing_info = load_data()
        
        # Convert to 3D format for CSP if needed
        n_channels = len(preprocessing_info.get('channels', [])) if preprocessing_info else 64
        X_3d = reshape_to_3d(X, n_channels) if n_channels > 0 else None
        
        # Load or train pipeline based on mode
        if args.mode in ['train', 'both']:
            # Load best pipeline from optimization
            pipeline, pipeline_info = load_best_pipeline()
            
            if pipeline is None:
                logger.error("Could not load best pipeline. Exiting.")
                return
                
            # Train the pipeline with all data
            is_csp = pipeline_info.get('is_csp', False) or 'CSP' in pipeline_info.get('best_pipeline_name', '')
            trained_pipeline, training_info = train_pipeline(
                pipeline, X, y, X_3d, test_size=args.test_size, is_csp=is_csp
            )
            
            if trained_pipeline is None:
                logger.error("Training failed. Exiting.")
                return
                
            pipeline = trained_pipeline
            is_csp = training_info.get('pipeline_type') in ['CSP', 'Ensemble']
            
        elif args.mode == 'predict':
            # Load the latest trained pipeline
            pipeline_path = os.path.join(models_dir, 'final_model_latest.joblib')
            info_path = os.path.join(models_dir, 'best_pipeline_info.json')
            
            try:
                pipeline = load(pipeline_path)
                logger.info(f"Loaded trained pipeline from {pipeline_path}")
                
                # Load pipeline info
                if os.path.exists(info_path):
                    with open(info_path, 'r') as f:
                        pipeline_info = json.load(f)
                    is_csp = pipeline_info.get('is_csp', False) or 'CSP' in pipeline_info.get('best_pipeline_name', '')
                else:
                    is_csp = False  # Default to standard pipeline if no info is available
                    
            except Exception as e:
                logger.error(f"Could not load trained pipeline: {e}")
                return
        
        # Run predictions if in predict or both modes
        if args.mode in ['predict', 'both']:
            # Simulate real-time prediction
            predict_realtime(
                pipeline, X, X_3d, y, 
                is_csp=is_csp,
                visualize=args.visualize,
                delay=args.delay
            )
            
    except Exception as e:
        logger.error(f"Error in main function: {e}")
        raise

if __name__ == "__main__":
    main()
